# DANN objective
Written only; no cells executed. See task2/README.md for execution order.

In [ ]:
"""Task 2 DANN: adversarial alignment of 512-D ResNet-18 features."""
import math

from torch import nn
from torch.autograd import Function


class _GradientReversal(Function):
    @staticmethod
    def forward(ctx, features, strength):
        ctx.strength = float(strength)
        return features.view_as(features)

    @staticmethod
    def backward(ctx, gradient):
        return -ctx.strength * gradient, None


def reverse_gradient(features, strength):
    """Identity forward; negate and scale only the gradient entering features."""
    return _GradientReversal.apply(features, strength)


def reversal_strength(progress, maximum=1.0):
    """p is progress through the fixed maximum training budget, not early stopping.

    alpha(p) = maximum * (2 / (1 + exp(-10*p)) - 1).
    The standard schedule approaches (rather than exactly reaches) maximum.
    """
    if not 0 <= progress <= 1:
        raise ValueError('Training progress must be in [0, 1].')
    return float(maximum) * (2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0)


def adversarial_objective(model, x, y, target_x, discriminator, cfg, progress, rng_state, conditional=False):
    source_features = forward_features(model, x)
    target_features = forward_features(model, target_x)
    logits = model.fc(source_features)
    classification = nn.functional.cross_entropy(logits, y)
    features = torch.cat((source_features, target_features), dim=0)
    if conditional:
        probabilities = torch.cat((logits, model.fc(target_features)), dim=0).softmax(dim=1)
        features = conditional_features(features, probabilities)
    alpha = reversal_strength(progress, cfg['max_grl_strength'])
    labels = torch.cat((torch.zeros(len(x), dtype=torch.long, device=x.device),
                        torch.ones(len(target_x), dtype=torch.long, device=x.device)))
    with torch.random.fork_rng(devices=[]):
        torch.set_rng_state(rng_state)
        domain_logits = discriminator(reverse_gradient(features, alpha))
        rng_state = torch.get_rng_state()
    alignment = nn.functional.cross_entropy(domain_logits, labels)
    correct = (domain_logits.argmax(1) == labels).sum().item()
    return logits, classification + alignment, classification, alignment, correct, len(labels), alpha, rng_state

def dann_objective(model, x, y, target_x, discriminator, cfg, progress, rng_state):
    return adversarial_objective(model, x, y, target_x, discriminator, cfg, progress, rng_state)